In [1]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import os
import nested_pandas as npd
import pyarrow as pa
from progressbar import progressbar
import pyarrow.parquet as pq
import pyarrow.compute as pc

from astropy.coordinates import SkyCoord
import astropy.units as u

## Separate only fields in common with feature analysis

In [2]:
used_fields = ['436', '512', '594', '641', '686', '705', '727', '770', '778', '805']

In [3]:
dirname = ['/media/snad/ztf_features/dr24-features_astra-clr_pretrained/embed_no_lc/' + \
          'ztfdr24_astra_embeddings/dataset/Norder=' +  str(i) +'/' for i in range(4, 8) ]

In [4]:
k = 2
flist = os.listdir(dirname[k])
flist_ = [os.listdir(dirname[k] + '/' + flist[j]) for j in range(len(flist))]

In [5]:
len(flist_[1])

644

In [6]:
count = list(np.arange(0, len(flist_[1]), 50)) + [len(flist_[1])]

In [8]:
# This cell checks the distance between all objects and Masha's given point for double checking
c1 = SkyCoord(ra=246.6812277*u.deg, dec=54.9451809*u.deg, frame='icrs')

index = 1

fname = flist[index]
fname_ = flist_[index]

for cc in progressbar(range(0, len(count) - 2)):
    data_list = []

    for c in range(count[cc], count[cc] + 1):

        dist_list = []
        data_temp = pd.read_parquet(dirname[k] + '/' + fname + '/' + fname_[c])

        data_temp['ra'] = data_temp['matches'].apply(lambda x: np.mean(x['objra']))
        data_temp['dec'] = data_temp['matches'].apply(lambda x: np.mean(x['objdec']))

        for i in range(data_temp.shape[0]):
            c2 = SkyCoord(ra=data_temp.iloc[i]['ra']*u.deg, dec=data_temp.iloc[i]['dec']*u.deg, frame='icrs')
            dist = c1.separation(c2).deg
            dist_list.append(dist)

        data_temp['distance_masha'] = dist_list
        data_list.append(data_temp)

    data_temp2 = pd.concat(data_list, ignore_index=True)
    data_temp2.to_parquet('data/Norder=' + str(k) + fname + '_' + str(count[cc]) + '_' + str(count[cc] + 1) + '.parquet')

    order = np.argsort(data_temp2['distance_masha'].values)
    print(data_temp2.iloc[order[0]][['ra','dec', 'distance_masha']])


  8% (1 of 12) |##                       | Elapsed Time: 0:00:00 ETA:   0:00:00

ra                275.899109
dec                 1.760434
distance_masha     58.251255
Name: 0, dtype: object
ra                290.954834
dec                -0.007247
distance_masha     65.723129
Name: 0, dtype: object


 25% (3 of 12) |######                   | Elapsed Time: 0:00:34 ETA:   0:01:44

ra                281.248138
dec                  9.59108
distance_masha     52.933222
Name: 18194, dtype: object


 33% (4 of 12) |########                 | Elapsed Time: 0:00:37 ETA:   0:01:14

...

ra                273.719513
dec                -0.849645
distance_masha     60.040406
Name: 1449, dtype: object


 83% (10 of 12) |####################    | Elapsed Time: 0:02:03 ETA:   0:00:24

ra                294.609558
dec                11.412149
distance_masha     57.369475
Name: 47421, dtype: object
ra                297.936279
dec                13.897242
distance_masha     56.937285
Name: 1, dtype: object


100% (12 of 12) |########################| Elapsed Time: 0:02:10 Time:  0:02:100:11


ra                289.615845
dec                 10.74156
distance_masha     55.548429
Name: 4865, dtype: object
ra                282.774017
dec                20.621424
distance_masha     43.723182
Name: 0, dtype: object


In [10]:
data_field = pd.concat(data_list, ignore_index=True)

In [11]:
data_field.to_parquet('data/fields_Norder7.parquet')

## Join all the common objects in the same file

In [3]:
flist = os.listdir('data/')

In [41]:
data_list = []
for fname in flist:
    table = pq.read_table("data/" + fname)
    
    target_type = pa.list_(pa.float32())
    for col in ["embedding_beggining", "embedding_middle", "embedding_end"]:
        table = table.set_column(table.schema.get_field_index(col), col,
                                  pc.cast(table[col], target_type))
    data_temp = table.to_pandas()

    data_list.append(data_temp)
data_fields = pd.concat(data_list, ignore_index=True)

In [42]:
data_fields.shape

(878132, 6)

### Average over embeddings

In [43]:
def embed_to_numpy(df, col_name, NDIM=512):
    pa_array = pa.array(df[col_name])
    np_1d_array = pa_array.values.to_numpy(zero_copy_only=False)
    np_2d_array = np_1d_array.reshape(-1, NDIM)
    return np_2d_array

In [44]:
emb_beg = embed_to_numpy(data_fields, "embedding_beggining")
emb_mid = embed_to_numpy(data_fields, "embedding_middle")
emb_end = embed_to_numpy(data_fields, "embedding_end")
emb_avg = 1./3. * (emb_beg + emb_mid + emb_end)


In [49]:
data_fields['embedding_average'] = list(emb_avg)

## Use only extragalactic fields

In [2]:
data_fields = pd.read_parquet('data/embeddings_common_fields.parquet')

In [8]:
data_fields['ra'] = data_fields['matches'].apply(lambda x: np.mean(x['objra']))
data_fields['dec'] = data_fields['matches'].apply(lambda x: np.mean(x['objdec']))

In [9]:
galactic_fields = ['436', '512', '594', '641', '686', '705', '727', '770', '778', '805']
extrag_fields = ['252', '401', '468', '626', '673', '718', '759', '795', '797', '848']

In [30]:
dist_list = []

for i in progressbar(range(data_fields.shape[0])):
    c2 = SkyCoord(ra=data_fields.iloc[i]['ra']*u.deg, dec=data_fields.iloc[i]['dec']*u.deg, frame='icrs')
    dist = c1.separation(c2)
    dist_list.append(dist)

data_fields['distance_masha'] = dist_list

100% (878132 of 878132) |################| Elapsed Time: 0:10:31 Time:  0:10:310551


In [36]:
order = np.argsort(data_fields['distance_masha'].values)

In [38]:
data_fields.iloc[order[0]]

koid                                                     727216200031198
matches                {'objra': [282.32803, 282.32797, 282.32794, 28...
embedding_beggining    [0.29614258, 0.39379883, 3.1308594, -4.2851562...
embedding_middle       [0.34887695, 0.51464844, 3.2246094, -4.3515625...
embedding_end          [0.07287598, -0.1583252, 2.25, -4.3828125, -0....
_healpix_29                                          1052688969472072826
embedding_average      [0.23929851, 0.2500407, 2.8684897, -4.3398438,...
ra                                                            282.328003
dec                                                            44.225941
distance_masha                                         25d07m07.4396948s
Name: 161178, dtype: object

In [39]:
data_fields.iloc[order[-1]]

koid                                                     512101100017505
matches                {'objra': [96.23989, 96.23988, 96.23986, 96.23...
embedding_beggining    [-0.29248047, -1.0361328, 2.8261719, -4.777343...
embedding_middle       [-0.18811035, -1.1484375, 2.8730469, -5.027343...
embedding_end          [-0.09552002, -0.3840332, 2.8300781, -5.054687...
_healpix_29                                          1663660049720269940
embedding_average      [-0.19203696, -0.8562012, 2.843099, -4.953125,...
ra                                                             96.239868
dec                                                             9.067813
distance_masha                                       111d22m03.15583909s
Name: 51253, dtype: object

In [70]:
data_use.shape

(0, 7)